## **El "Por Qué" del Seguimiento de Experimentos**

En ML, a menudo realizamos muchos experimentos para encontrar el mejor modelo para una tarea dada. Un experimento puede involucrar:

*   Probar diferentes arquitecturas de modelos (por ejemplo, RandomForest, GradientBoosting).
*   Ajustar hiperparámetros (por ejemplo, `max_depth`, `n_estimators`).
*   Usar diferentes conjuntos de características (features).
*   Variar las técnicas de preprocesamiento.

Llevar un registro de todos estos experimentos puede volverse rápidamente un proceso desordenado y propenso a errores. Aquí es donde entra en juego el seguimiento de experimentos.

## **¿Qué es el Experiment Tracking?**

El experiment tracking es el proceso de registrar y organizar de manera sistemática toda la información relacionada con tus experimentos de machine learning. Esto incluye:
*	Código: La versión exacta del código utilizada para ejecutar el experimento.
*	Datos: La versión del dataset empleado.
*	Parámetros: Los hiperparámetros y otras configuraciones.
*	Métricas: Las métricas de desempeño del modelo (por ejemplo, RMSE, accuracy).
*	Artefactos: El modelo entrenado, las visualizaciones y otros archivos generados.


## **¿Por qué es Importante?**

Reproducibilidad: Reproducir fácilmente resultados anteriores, lo cual es crucial para depuración (debugging) y para construir sobre trabajos previos.

* Organización: Mantener tus experimentos organizados y evitar perder el rastro de lo que ya has probado.
* Colaboración: Compartir tus resultados con colegas y colaborar de manera más efectiva.
* Comparación: Comparar fácilmente el desempeño de diferentes modelos y experimentos.
* Deployment: Hacer seguimiento de los modelos que se despliegan en producción y de su rendimiento.

## **Herramientas para Experiment Tracking**

Existen varias herramientas disponibles para experiment tracking, tanto de código abierto como comerciales. Algunas de las más populares incluyen:
* MLflow: Una plataforma open-source para el ciclo de vida de machine learning, incluyendo experiment tracking.
* Weights & Biases: Una plataforma comercial para experiment tracking y colaboración.
* Comet: Otra plataforma comercial con foco en experiment tracking y monitoreo de modelos.
* TensorBoard: Un kit de visualización para TensorFlow que también incluye funcionalidades de experiment tracking.

En este curso, nos enfocaremos en MLflow.

## **Training con MLFlow**

In [ ]:
import pandas as pd

## **Cargamos datos "procesados"**

In [2]:
df_jan = pd.read_parquet("data/processed/jan.parquet")
df_feb = pd.read_parquet("data/processed/feb.parquet")


## **Definir columnas a modelar**

In [3]:
categorical = ['PULocationID', 'DOLocationID']
numerical = ["trip_distance"]

## Definir set de train y test

In [4]:
df_train = df_jan
df_val = df_feb

## **Preprocesar**

In [5]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

df_train = df_train.copy()
df_val   = df_val.copy()
df_train["categorical_dict"] = df_train[categorical].to_dict(orient="records")
df_val["categorical_dict"]   = df_val[categorical].to_dict(orient="records")

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", DictVectorizer(), "categorical_dict"),
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                          ("scaler", StandardScaler())]), numerical),
    ]
)

X_train = preprocessor.fit_transform(df_train)
y_train = df_train["duration"].values
X_val   = preprocessor.transform(df_val)
y_val   = df_val["duration"].values

## **Entrenar**

## **Antes de entrenar: levanta el MLflow Tracking Server**

En una terminal, en la raíz del proyecto (misma carpeta donde estás ejecutando este notebook), corre:

```bash
mlflow server \
  --host 127.0.0.1 \
  --port 5001 \
  --backend-store-uri sqlite:///mlflow.db \
  --default-artifact-root ./mlruns
```

Luego abre la UI en: http://127.0.0.1:5000

In [ ]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5001")
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

tracking URI: 'http://127.0.0.1:5000'


In [7]:
mlflow.search_experiments()

[<Experiment: artifact_location='/Users/mdurango/Downloads/proyectos/MLOps_UdM/02-Experiment-Tracking/notebooks/mlruns/0', creation_time=1774486367857, experiment_id='0', last_update_time=1774486367857, lifecycle_stage='active', name='Default', tags={}>]

In [14]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

experiment_name = "nyc-taxi-tracking"
mlflow.set_experiment(experiment_name)

n_estimators = 150
max_depth = 7


with mlflow.start_run(run_name=f"rf_depth{max_depth}_trees{n_estimators}"):
    # Tags (metadata)
    mlflow.set_tag("problem_type", "regression")
    mlflow.set_tag("model_family", "random_forest")
    mlflow.set_tag("dataset", "nyc_green_taxi_2023_01_02")
    mlflow.set_tag("features", "PULocationID,DOLocationID,trip_distance")

    # Params
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("random_state", 0)

    # Train + predict
    rf = RandomForestRegressor(n_estimators=n_estimators, random_state=0, max_depth=max_depth)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_val)

    # Metrics
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mae = mean_absolute_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)

    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)

    # Artifacts: predictions table
    preds_path = "predictions_rf.csv"
    pd.DataFrame({"y_true": y_val, "y_pred": y_pred, "error": y_val - y_pred}).to_csv(preds_path, index=False)
    mlflow.log_artifact(preds_path)

    # Artifact: feature importance
    import matplotlib.pyplot as plt

    importances = rf.feature_importances_
    top_idx = np.argsort(importances)[-20:]

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(range(len(top_idx)), importances[top_idx])
    ax.set_yticks(range(len(top_idx)))
    ax.set_yticklabels([str(i) for i in top_idx])
    ax.set_title("Top-20 feature importances (by encoded index)")
    fig.tight_layout()

    fig_path = "feature_importance_rf.png"
    fig.savefig(fig_path, dpi=150)
    mlflow.log_artifact(fig_path)
    plt.close(fig)

    # Artifact: residual histogram
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(y_val - y_pred, bins=50)
    ax.set_title("Residuals (y_true - y_pred)")
    ax.set_xlabel("Residual")
    ax.set_ylabel("Count")
    fig.tight_layout()

    fig_path = "residuals_rf.png"
    fig.savefig(fig_path, dpi=150)
    mlflow.log_artifact(fig_path)
    plt.close(fig)

    rmse

2026/03/25 20:37:50 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run rf_depth7_trees150 at: http://127.0.0.1:5000/#/experiments/1/runs/253d4fb19b4f4005963024556dec94ce
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


## **Probando otro modelo**

In [15]:
import xgboost as xgb

with mlflow.start_run(run_name="xgb_default"):
    mlflow.set_tag("problem_type", "regression")
    mlflow.set_tag("model_family", "xgboost")
    mlflow.set_tag("dataset", "nyc_green_taxi_2023_01_02")
    mlflow.set_tag("features", "PULocationID,DOLocationID,trip_distance")

    xgb_reg = xgb.XGBRegressor(objective="reg:squarederror", random_state=42)
    xgb_reg.fit(X_train, y_train)
    y_pred = xgb_reg.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mae = mean_absolute_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)

    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)

    rmse

2026/03/25 20:37:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/25 20:37:56 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/mdurango/Downloads/proyectos/MLOps_UdM/.venv/lib/python3.11/site-packages/xgboost/sklearn.py:1028: UserWarning: [20:37:56] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats."
2026/03/25 20:37:57 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run xgb_default at: http://127.0.0.1:5000/#/experiments/1/runs/6bee017d0da449799337ffe29e25be39
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


## **Probando autologging (y combinándolo con logs custom)**

`mlflow.autolog()` ayuda a loggear automáticamente muchos detalles (parámetros, métricas, modelo, etc.).
Aún así, es normal complementar con `mlflow.set_tag(...)` y con artifacts custom (plots/tablas).

In [11]:
import mlflow

experiment_name = "nyc-taxi-autolog"
mlflow.set_experiment(experiment_name)

mlflow.autolog()

with mlflow.start_run(run_name="rf_autolog_depth10"):
    mlflow.set_tag("problem_type", "regression")
    mlflow.set_tag("model_family", "random_forest")
    mlflow.set_tag("note", "Autolog enabled; we still add tags and a custom artifact")

    rf = RandomForestRegressor(n_estimators=100, random_state=0, max_depth=10)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mlflow.log_metric("rmse_manual", rmse)

    # Custom artifact even with autolog
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.scatter(y_val, y_pred, s=5, alpha=0.3)
    ax.set_title("y_true vs y_pred")
    ax.set_xlabel("y_true")
    ax.set_ylabel("y_pred")
    fig.tight_layout()

    fig_path = "true_vs_pred.png"
    fig.savefig(fig_path, dpi=150)
    mlflow.log_artifact(fig_path)
    plt.close(fig)

    rmse

2026/03/25 20:32:46 INFO mlflow.tracking.fluent: Experiment with name 'nyc-taxi-autolog' does not exist. Creating a new experiment.
2026/03/25 20:32:49 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/03/25 20:32:49 WARNING mlflow.utils.autologging_utils: MLflow xgboost autologging is known to be compatible with 2.0.0 <= xgboost <= 3.0.2, but the installed version is 3.0.4. If you encounter errors during autologging, try upgrading / downgrading xgboost to a compatible version, or try upgrading MLflow.
2026/03/25 20:32:49 INFO mlflow.tracking.fluent: Autologging successfully enabled for xgboost.
2026/03/25 20:32:54 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run rf_autolog_depth10 at: http://127.0.0.1:5000/#/experiments/2/runs/dcc8bc7d62934113af82b4f0568483da
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [12]:
import mlflow
import pandas as pd

runs = mlflow.search_runs(
    experiment_names=["nyc-taxi-tracking"],
    order_by=["metrics.rmse ASC"],
    max_results=5,
)

runs[[
    "run_id",
    "metrics.rmse",
    "metrics.mae",
    "metrics.r2",
    "tags.model_family",
]]

,run_id,metrics.rmse,metrics.mae,metrics.r2,tags.model_family
0,58706a778aa3410bb9738a37bae28342,4.978182,3.168270,0.714606,xgboost
1,c089c3af25d6437c878b19215d7edc13,5.137714,3.294793,0.696021,random_forest
2,d2b3fa8a5de74ff98c21c995ce480c16,5.357379,3.468562,0.669472,random_forest
